# Phase 1: Experiment EDA

Goal: validate the randomized advertising experiment before building uplift models.

By the end of this notebook, we should know:

- How many users are in the experiment
- Whether treatment/control groups are the right size
- Baseline conversion rates by treatment group
- Average treatment effect (ATE)
- Whether randomized groups are balanced on user/context features
- Whether the data is clean enough for Phase 2 uplift modeling

## 1. Setup

We use DuckDB to query the raw CSV directly. This avoids loading the full 3GB file into pandas memory.

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
# Works whether VS Code starts the notebook from the project root or notebooks/.
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "criteo-uplift-v2.1.csv"

assert DATA_PATH.exists(), f"Missing data file: {DATA_PATH}"
DATA_PATH

In [ ]:
con = duckdb.connect(database=":memory:")

# sample_size controls CSV type inference only. It does not limit the rows in the view.
# Queries against experiment still scan the full raw dataset unless you add limit or using sample.
con.execute(f"""
    create or replace view experiment as
    select *
    from read_csv_auto('{DATA_PATH.as_posix()}', sample_size=100000)
""")

con.sql("select * from experiment limit 5").df()

## 2. Schema and Experiment Size

In [ ]:
schema = con.sql("describe experiment").df()
schema

In [ ]:
columns = schema["column_name"].tolist()
feature_cols = [c for c in columns if c.startswith("f")]

TREATMENT_COL = "treatment"
OUTCOME_COL = "conversion"
VISIT_COL = "visit"
EXPOSURE_COL = "exposure"

print(f"Columns: {len(columns)}")
print(f"Feature columns: {feature_cols}")

In [ ]:
con.sql("select count(*) as users from experiment").df()

## 3. Treatment and Outcome Summary

This is the core product analytics table: treatment size, control size, and raw conversion rates.

In [ ]:
group_summary = con.sql(f"""
    select
        {TREATMENT_COL} as treatment,
        count(*) as users,
        count(*) * 1.0 / sum(count(*)) over () as user_share,
        avg({OUTCOME_COL}) as conversion_rate,
        avg({VISIT_COL}) as visit_rate,
        avg({EXPOSURE_COL}) as exposure_rate
    from experiment
    group by 1
    order by 1
""").df()

group_summary

In [ ]:
treatment_stats = group_summary.set_index("treatment")

control_cr = treatment_stats.loc[0, "conversion_rate"]
treatment_cr = treatment_stats.loc[1, "conversion_rate"]

ate = treatment_cr - control_cr
relative_lift = ate / control_cr

pd.DataFrame({
    "metric": [
        "control_conversion_rate",
        "treatment_conversion_rate",
        "ate_percentage_points",
        "relative_lift",
    ],
    "value": [control_cr, treatment_cr, ate, relative_lift],
})

In [ ]:
# Difference-in-proportions confidence interval for ATE.
n_t = treatment_stats.loc[1, "users"]
n_c = treatment_stats.loc[0, "users"]
p_t = treatment_cr
p_c = control_cr

se = np.sqrt((p_t * (1 - p_t) / n_t) + (p_c * (1 - p_c) / n_c))
ci_low = ate - 1.96 * se
ci_high = ate + 1.96 * se
z_score = ate / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

pd.DataFrame({
    "ate": [ate],
    "std_error": [se],
    "ci_low": [ci_low],
    "ci_high": [ci_high],
    "p_value": [p_value],
})

### Interpretation

After running the cells above, write 2-3 sentences:

- Is treatment associated with higher conversion?
- How large is the lift in percentage points and relative terms?
- Is the result practically meaningful for a targeting/promotion decision?

## 4. Missing Values

In [ ]:
missing_exprs = ",\n        ".join([
    f"sum(case when {col} is null then 1 else 0 end) as {col}_missing"
    for col in columns
])

missing_wide = con.sql(f"""
    select
        count(*) as users,
        {missing_exprs}
    from experiment
""").df()

missing = (
    missing_wide.drop(columns="users")
    .T
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)

missing["column"] = missing["column"].str.replace("_missing", "", regex=False)
missing["missing_rate"] = missing["missing_count"] / missing_wide.loc[0, "users"]
missing.sort_values("missing_rate", ascending=False)

## 5. Randomization Balance Check

For a randomized experiment, pre-treatment features should look similar across treatment and control.

We use standardized mean difference (SMD):

`SMD = (mean_treatment - mean_control) / pooled_standard_deviation`

Rule of thumb: `abs(SMD) < 0.1` is usually considered balanced.

In [ ]:
balance_rows = []

for col in feature_cols:
    stats_by_group = con.sql(f"""
        select
            {TREATMENT_COL} as treatment,
            avg({col}) as mean_value,
            stddev_samp({col}) as sd_value
        from experiment
        group by 1
        order by 1
    """).df().set_index("treatment")

    mean_c = stats_by_group.loc[0, "mean_value"]
    mean_t = stats_by_group.loc[1, "mean_value"]
    sd_c = stats_by_group.loc[0, "sd_value"]
    sd_t = stats_by_group.loc[1, "sd_value"]
    pooled_sd = np.sqrt((sd_t ** 2 + sd_c ** 2) / 2)
    smd = (mean_t - mean_c) / pooled_sd if pooled_sd else np.nan

    balance_rows.append({
        "feature": col,
        "control_mean": mean_c,
        "treatment_mean": mean_t,
        "standardized_mean_difference": smd,
        "abs_smd": abs(smd),
    })

balance = pd.DataFrame(balance_rows).sort_values("abs_smd", ascending=False)
balance

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(
    data=balance.sort_values("abs_smd", ascending=True),
    x="abs_smd",
    y="feature",
    color="#4C78A8",
)
plt.axvline(0.1, color="#D62728", linestyle="--", label="0.1 threshold")
plt.title("Covariate Balance by Absolute Standardized Mean Difference")
plt.xlabel("Absolute standardized mean difference")
plt.ylabel("Feature")
plt.legend()
plt.tight_layout()

### Interpretation

After running the cells above, write 2-3 sentences:

- Do treatment and control look balanced?
- Are any features above the 0.1 SMD threshold?
- Does this support using the experiment for causal uplift modeling?

## 6. Treatment Assignment vs Exposure

Important causal note: treatment assignment is randomized. Exposure may happen after assignment, so avoid treating exposure as if it were randomly assigned.

In [ ]:
con.sql(f"""
    select
        {TREATMENT_COL} as treatment,
        {EXPOSURE_COL} as exposure,
        count(*) as users,
        avg({OUTCOME_COL}) as conversion_rate,
        avg({VISIT_COL}) as visit_rate
    from experiment
    group by 1, 2
    order by 1, 2
""").df()

## 7. Row-Level Sample for Lightweight Plots

Use the full dataset for aggregate SQL metrics. Use samples for row-level visualizations.

In [ ]:
sample_df = con.sql("""
    select *
    from experiment
    using sample 100000 rows
""").df()

sample_df.shape

In [ ]:
plot_features = feature_cols[:4]

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.flatten()

for ax, col in zip(axes, plot_features):
    sns.kdeplot(
        data=sample_df,
        x=col,
        hue=TREATMENT_COL,
        common_norm=False,
        ax=ax,
    )
    ax.set_title(f"Distribution of {col} by Treatment")

plt.tight_layout()

## 8. Phase 1 Readout

Fill this in after running the notebook:

- The experiment contains `[N]` users, with `[X%]` assigned to treatment.
- Treatment conversion rate was `[A%]` versus control conversion rate of `[B%]`.
- The average treatment effect was `[C]` percentage points, or `[D%]` relative lift.
- Covariate balance looked `[good / concerning]`, with max absolute SMD of `[E]`.
- Phase 1 conclusion: this experiment is `[suitable / not suitable yet]` for heterogeneous treatment effect modeling.

Next step: build uplift models to estimate which users are persuadable, neutral, or negatively affected.